[Reference](https://medium.com/@anubhavgoyal101/530fbb25b613)

#  Fixed-size chunking

In [1]:
def chunk_fixed(text: str, size: int = 400, overlap: int = 50) -> list[str]:
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(" ".join(words[i : i + size]))
        i += size - overlap
    return chunks

# Recursive chunking

In [2]:
import re

def chunk_recursive(text: str, max_tokens: int = 512) -> list[str]:
    separators = [
        r"\n#{1,6}\s",   # headings
        r"\n\n",          # paragraphs
        r"(?<=\.)\s",     # sentences
    ]
    return _split_recursive(text, separators, max_tokens)

def _split_recursive(text: str, seps: list[str], max_t: int) -> list[str]:
    if len(text.split()) <= max_t or not seps:
        return [text.strip()] if text.strip() else []

    parts = re.split(seps[0], text)
    chunks = []
    for p in parts:
        if len(p.split()) <= max_t:
            chunks.append(p.strip())
        else:
            chunks.extend(_split_recursive(p, seps[1:], max_t))
    return [c for c in chunks if c]

# Structure-aware chunking

In [3]:
HEADING = re.compile(r"^(#{1,6})\s+(.+)$", re.MULTILINE)
_CODE = re.compile(r"```[\s\S]*?```")
_WARNING = re.compile(r"^>\s*\*\*(Warning|Note|Caution)\*\*.*", re.MULTILINE)

from dataclasses import dataclass

@dataclass
class Chunk:
    content: str
    heading_path: list[str]
    doc_id: str
    meta: dict
    has_code: bool = False
    has_table: bool = False

def chunk_structured(text: str, max_tokens: int = 512) -> list[Chunk]:
    parts = _HEADING.split(text)
    chunks = []
    stack = []

    if parts[0].strip():
        chunks.append(Chunk(
            content=parts[0].strip(), heading_path=[], doc_id="",
            meta={}, has_code=bool(_CODE.search(parts[0])),
        ))

    for i in range(1, len(parts) - 1, 3):
        level = len(parts[i])
        title = parts[i + 1].strip()
        body = parts[i + 2].strip() if i + 2 < len(parts) else ""

        if not body:
            continue

        while stack and stack[-1][0] >= level:
            stack.pop()
        stack.append((level, title))
        path = [h[1] for h in stack]

        sections = _split_preserving_warnings(body, max_tokens)
        for sec in sections:
            chunks.append(Chunk(
                content=sec, heading_path=path, doc_id="", meta={},
                has_code=bool(_CODE.search(sec)),
                has_table=bool("|" in sec and "---" in sec),
            ))
    return chunks

def _split_preserving_warnings(text: str, max_t: int) -> list[str]:
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    merged = []
    i = 0
    while i < len(paras):
        block = paras[i]
        while i + 1 < len(paras) and _WARNING.match(paras[i + 1]):
            block += "\n\n" + paras[i + 1]
            i += 1
        merged.append(block)
        i += 1

    result = []
    buf = []
    buf_t = 0
    for m in merged:
        t = len(m.split())
        if buf and buf_t + t > max_t:
            result.append("\n\n".join(buf))
            buf = []
            buf_t = 0
        buf.append(m)
        buf_t += t
    if buf:
        result.append("\n\n".join(buf))
    return result

In [4]:
from dataclasses import dataclass

@dataclass
class EvalCase:
    question: str
    expected_chunk_contains: str
    category: str

def eval_chunking(
    text: str,
    cases: list[EvalCase],
    strategies: dict[str, callable],
    embed_fn: callable,
    search_fn: callable,
    top_k: int = 3,
) -> dict[str, dict]:
    results = {}

    for name, chunk_fn in strategies.items():
        chunks = chunk_fn(text)
        contents = [c.content if hasattr(c, 'content') else c for c in chunks]
        chunk_embs = embed_fn(contents)

        hits = 0
        total = len(cases)
        for case in cases:
            q_emb = embed_fn([case.question])[0]
            idxs = search_fn(q_emb, chunk_embs, top_k)
            retrieved = [contents[i] for i in idxs]

            if any(case.expected_chunk_contains in r for r in retrieved):
                hits += 1

        results[name] = {
            "hit_rate": hits / total if total else 0,
            "num_chunks": len(chunks),
            "avg_tokens": sum(len(c.split()) for c in contents) // max(len(contents), 1),
        }
    return results